Allows import of utils from root directory

In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

In [2]:
import pandas as pd
price_df = pd.read_csv("data/prices_round_0_day_-2.csv" , sep = ';')
price_df.head(5)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-2,0,EMERALDS,9992,11,9990,25,NaN,NaN,10008,11,10010,25,NaN,NaN,10000.0,0.0
1,-2,0,TOMATOES,4993,7,4992,17,NaN,NaN,5007,7,5008,17,NaN,NaN,5000.0,0.0
2,-2,100,TOMATOES,4998,5,4993,7,4992.0,16.0,5007,7,5008,16,NaN,NaN,5002.5,0.0
3,-2,100,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0
4,-2,200,TOMATOES,4994,6,4993,20,NaN,NaN,5008,6,5009,20,NaN,NaN,5001.0,0.0


In [3]:
trade_df = pd.read_csv("data/trades_round_0_day_-2.csv" , sep = ';')
trade_df.head(5)

,timestamp,buyer,seller,symbol,currency,price,quantity
0,900,NaN,NaN,TOMATOES,XIRECS,5008.0,2
1,1700,NaN,NaN,TOMATOES,XIRECS,5006.0,3
2,4000,NaN,NaN,EMERALDS,XIRECS,10008.0,7
3,4100,NaN,NaN,TOMATOES,XIRECS,5002.0,3
4,5200,NaN,NaN,EMERALDS,XIRECS,9992.0,5


In [4]:
emerald_price_df = price_df[price_df['product']=='EMERALDS'].reset_index(drop=True)
emerald_trade_df = trade_df[trade_df['symbol']=='EMERALDS'].reset_index(drop=True)

In [5]:
from order_flow_analysis import detect_levels, level_coverage

detect_levels(emerald_price_df)
level_coverage(emerald_price_df)



Detected 3 order book levels

  Level       Bid %    Ask %     Use?
  ------------------------------------
  L1         100.0%   100.0%        ✅
  L2         100.0%   100.0%        ✅
  L3           1.6%     1.7%        ❌

  Recommended n_levels: 2


({1: (np.float64(100.0), np.float64(100.0), np.True_),
  2: (np.float64(100.0), np.float64(100.0), np.True_),
  3: (np.float64(1.63), np.float64(1.7000000000000002), np.False_)},
 2)

As order book level 1 and 2 are filled the most we will use micro price 2

In [6]:
from order_flow_analysis import mid_price, multi_level_micro_price
from stat_utils import compute_returns

emerald_price_df['mid_price'] = mid_price(emerald_price_df)
emerald_price_df['micro_price'] = multi_level_micro_price(emerald_price_df, 2)
returns_t = compute_returns(emerald_price_df['micro_price'])

In [7]:
from stat_utils import stationarity_panel

print(stationarity_panel(emerald_price_df["mid_price"]))
print(stationarity_panel(emerald_price_df["micro_price"]))
print(stationarity_panel(returns_t["R_t"]))
print(stationarity_panel(returns_t["r_t"]))

/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-71.87660082416127), 'adf_pvalue': 0.0, 'adf_usedlag': 1, 'kpss_stat': np.float64(0.11136551637013818), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 2}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-98.70857435058194), 'adf_pvalue': 0.0, 'adf_usedlag': 0, 'kpss_stat': np.float64(0.2223745752506001), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 0}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-28.016045549209142), 'adf_pvalue': 0.0, 'adf_usedlag': 38, 'kpss_stat': np.float64(0.04687850256980733), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 940}
{'adf_stat': np.float64(-28.01560142651875), 'adf_pvalue': 0.0, 'adf_usedlag': 38, 'kpss_stat': np.float64(0.04687833823430019), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 940}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


In [8]:
from stat_utils import distribution_summary

print(distribution_summary(emerald_price_df["mid_price"]))
print(distribution_summary(emerald_price_df["micro_price"]))
print(distribution_summary(returns_t["R_t"]))
print(distribution_summary(returns_t["r_t"]))

{'mean': np.float64(9999.9972), 'variance': np.float64(0.5328454445444544), 'skewness': np.float64(-0.10370451432480296), 'kurtosis': np.float64(30.04335424533479), 'jb_stat': np.float64(304426.051832077), 'jb_pvalue': np.float64(0.0)}
{'mean': np.float64(9999.999233496786), 'variance': np.float64(0.013311598427445436), 'skewness': np.float64(-0.8731682267470309), 'kurtosis': np.float64(76.45041123143285), 'jb_stat': np.float64(2246887.8081808616), 'jb_pvalue': np.float64(0.0)}
{'mean': np.float64(1.3140710997142648e-10), 'variance': np.float64(2.628407666589229e-10), 'skewness': np.float64(0.09213748604220082), 'kurtosis': np.float64(39.106322510780565), 'jb_stat': np.float64(542593.1845418566), 'jb_pvalue': np.float64(0.0)}
{'mean': np.float64(0.0), 'variance': np.float64(2.628403765036979e-10), 'skewness': np.float64(0.09121136364326868), 'kurtosis': np.float64(39.106156377479934), 'jb_stat': np.float64(542587.9085216303), 'jb_pvalue': np.float64(0.0)}


In [9]:
price_graph = go.Figure()
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['mid_price'], mode='lines', name='Mid Price'))
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['micro_price'], mode='lines', name='Micro Price'))

price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['bid_price_1'], mode='lines', name='Best Bid Price'))
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['bid_price_2'], mode='lines', name='Level 2 Bid Price'))
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['bid_price_3'], mode='markers', name='Level 3 Bid Price'))

price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['ask_price_1'], mode='lines', name='Best Ask Price'))
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['ask_price_2'], mode='lines', name='Level 2 Ask Price'))
price_graph.add_trace(go.Scatter(x=emerald_price_df.index, y=emerald_price_df['ask_price_3'], mode='markers', name='Level 3 Ask Price'))
price_graph.show()



Potential pattern identified above, when l3 order is put l1 and l2orders spread narrows, mid point price follows this trend, whilst micro price slightly moves

In [10]:
from order_flow_analysis import order_levels_pattern

order_levels_pattern(emerald_price_df)



Detected 3 order book levels

  Level       Bid %    Ask %     Use?
  ------------------------------------
  L1         100.0%   100.0%        ✅
  L2         100.0%   100.0%        ✅
  L3           1.6%     1.7%        ❌

  Recommended n_levels: 2
SECTION 1: L3 Coverage (deepest detected level)
  L3 bid only:  1.63%  (n=163)
  L3 ask only:  1.70%  (n=170)
  L3 both:      0.00%  (n=0)
  L3 either:    3.33%  (n=333)
  L3 absent:    96.67%  (n=9667)

SECTION 2: Spread Compression at L1 and L2 During L3 Events
  L1 spread (L3 either)                    present=    8.0000  absent=   16.0000  diff=   -8.0000  p=0.0000 ***
  L2 spread (L3 either)                    present=   18.0000  absent=   20.0000  diff=   -2.0000  p=0.0000 ***
  L1 spread (L3 bid only)                  present=    8.0000  absent=   16.0000  diff=   -8.0000  p=0.0000 ***
  L1 spread (L3 ask only)                  present=    8.0000  absent=   16.0000  diff=   -8.0000  p=0.0000 ***
  L1 spread (L3 both)                   

/home/user/projects/imc_prosperity_tutorial/.venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/home/user/projects/imc_prosperity_tutorial/order_flow_analysis.py:300: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = spearmanr(x[mask], y[mask])
/home/user/projects/imc_prosperity_tutorial/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  L3 binary flag                                     p=0.5713 (ns)
  L3 signal (bid vol - ask vol)                      p=0.0000 ***

PATTERN SUMMARY
  ✅ L1 spread compresses during L3
  ✅ Mid price moves more during L3
  ✅ Micro price stays stable during L3
  ✅ L2 volume stays stable during L3
  ✅ Mid moves toward micro during L3
  ❌ L3 volume correlates with compression
  ❌ L3 flag Granger-causes returns

  ⚠️  L3 IS REGIME FLAG ONLY — no temporal predictive power confirmed

  ⚠️  PATTERN PARTIALLY CONFIRMED


/home/user/projects/imc_prosperity_tutorial/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


This confrims that L3 never appears on both sides simultaneously (0 co-occurrence), this is the most important finding. L3 is always one-sided, meaning it's a deliberate directional order. L1 moves aggressively toward mid, L2 doesn't move at all. This definitively confirms L2 is the true price anchor and L1 is just reacting to the L3 order presence.

This confirms that it is structural bot behaviour not just chance, next is too check for spoofing.

In [11]:
from order_flow_analysis import analyse_deep_level_spoofing, calculate_baseline_fill_rate, analyse_cross_level_spoofing

print(calculate_baseline_fill_rate(emerald_price_df, emerald_trade_df, 1))
print(calculate_baseline_fill_rate(emerald_price_df, emerald_trade_df, 2))
print(calculate_baseline_fill_rate(emerald_price_df, emerald_trade_df, 3))

analyse_deep_level_spoofing(emerald_price_df, emerald_trade_df)
analyse_cross_level_spoofing(emerald_price_df, emerald_trade_df)

{'total': {'fill_rate': 0.022438815401917205, 'withdrawn': 18673, 'executed': 419}, 'bid': {'fill_rate': 0.023370497427101202, 'withdrawn': 9328, 'executed': 218}, 'ask': {'fill_rate': 0.021508828250401284, 'withdrawn': 9345, 'executed': 201}}
{'total': {'fill_rate': 0.0, 'withdrawn': 35255, 'executed': 0}, 'bid': {'fill_rate': 0.0, 'withdrawn': 17604, 'executed': 0}, 'ask': {'fill_rate': 0.0, 'withdrawn': 17651, 'executed': 0}}
{'total': {'fill_rate': 0.0, 'withdrawn': 8, 'executed': 0}, 'bid': {'fill_rate': 0.0, 'withdrawn': 6, 'executed': 0}, 'ask': {'fill_rate': 0.0, 'withdrawn': 2, 'executed': 0}}
SECTION 4: Spoofing Analysis (L3) | Bid vs Ask Breakdown
Side          Withdrawn     Executed   Fill Ratio
--------------------------------------------------------------------------------
TOTAL                 8            0        0.00%
BID                   6            0        0.00%
ASK                   2            0        0.00%
  VERDICT (BID): 🚨 SPOOFING
  VERDICT (ASK): 🚨 SPOOF

{'l1_bid_delta': -0.023370497427101202}

Though this sounds crazy it confirms the market is very heavily spoofed, which we can verify as the trade csv only has 600 lines, the cross level validation also tells, us that when l3 is spoofed, the number of l1 orders filled doubles, this tells us that the bot doing the spoofing does it when trying to fill there orders.